In [ ]:
!pip install -qU langgraph langchain-openai langchain-google-genai langchain-community

In [5]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
import os
import json
import subprocess
from typing import TypedDict, Annotated, List, Dict, Any
import operator

from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

# 1. LLM Մոդելի սկզբնավորում
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)


# 2. State-ի սահմանում
class AgentState(TypedDict):
    dialogue: Annotated[List[Dict[str, str]], operator.add]
    current_role: str
    instruction: str
    reason: str
    step_count: int


# 3. Executor Tool
def execute_python_code(code: str, filename: str = "pinn_silverbox.py") -> str:
    with open(filename, "w", encoding="utf-8") as f:
        f.write(code)

    try:
        result = subprocess.run(
            ["python3", filename],
            capture_output=True,
            text=True,
            timeout=90
        )
        if result.returncode == 0:
            return f"SUCCESS (Exit code 0):\nSTDOUT:\n{result.stdout}"
        else:
            return f"ERROR (Exit code {result.returncode}):\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
    except subprocess.TimeoutExpired:
        return "ERROR: Execution timed out (exceeded 90 seconds)."
    except Exception as e:
        return f"EXECUTION FAILED: {str(e)}"


# 4. Agent Nodes-ի սահմանում

def orchestrator_node(state: AgentState):
    step_count = state.get("step_count", 0) + 1

    if step_count >= 8:
        return {
            "current_role": "finish",
            "instruction": "Maximum steps reached. Stop pipeline.",
            "reason": "Step limit reached.",
            "step_count": step_count
        }

    sys_prompt = """You are the Orchestrator for Task 5 Silverbox PINN on SNLS80mV.mat.
Analyze dialogue history and select next agent: 'scientist', 'general', 'executor', or 'finish'.

Output ONLY valid JSON:
- "next_role": "scientist" | "general" | "executor" | "finish"
- "instruction": "Clear directive"
- "reason": "Single sentence"

Rules:
1. Select 'executor' to write and run PyTorch code.
2. Select 'finish' ONLY if Executor prints "REAL TEST RMSE: <value>" successfully."""

    history_str = json.dumps(state["dialogue"], indent=2)
    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Current Dialogue History:\n{history_str}")
    ])

    try:
        res_json = json.loads(response.content.strip().strip("```json").strip("```"))
        next_role = res_json.get("next_role", "finish")
        instruction = res_json.get("instruction", "")
        reason = res_json.get("reason", "")
    except Exception:
        next_role = "executor"
        instruction = "Execute PyTorch PINN code."
        reason = "Parsing fallback."

    return {
        "current_role": next_role,
        "instruction": instruction,
        "reason": reason,
        "step_count": step_count,
        "dialogue": [{"role": "Orchestrator", "content": f"Decision -> Role: {next_role} | Reason: {reason} | Instruction: {instruction}"}]
    }


def scientist_node(state: AgentState):
    sys_prompt = "You are the Scientist agent. Provide concise RK4 physics-informed neural network dynamics."
    instruction = state["instruction"]
    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Instruction: {instruction}")
    ])
    return {"dialogue": [{"role": "Scientist", "content": response.content}]}


def general_node(state: AgentState):
    sys_prompt = "You are the General agent. Answer high-level questions."
    instruction = state["instruction"]
    response = llm.invoke([
        SystemMessage(content=sys_prompt),
        HumanMessage(content=f"Instruction: {instruction}")
    ])
    return {"dialogue": [{"role": "General Agent", "content": response.content}]}


def executor_node(state: AgentState):
    # Ճշգրիտ, չվրիպող PyTorch կոդի տեքստ
    fixed_code = '''import numpy as np
import torch
import torch.nn as nn
import scipy.io

# 1. Data loading and preprocessing
mat = scipy.io.loadmat('SNLS80mV.mat')
u = mat['V1'].squeeze()[42650:46746].astype(np.float32)
y = mat['V2'].squeeze()[42650:46746].astype(np.float32)
u -= u.mean()
y -= y.mean()

dt = np.float32(1.0 / 610.35)
v = np.gradient(y, dt).astype(np.float32)

u_train, y_train, v_train = u[:3072], y[:3072], v[:3072]
u_test, y_test = u[3072:], y[3072:]

u_tr_t = torch.tensor(u_train).view(-1, 1)
y_tr_t = torch.tensor(y_train).view(-1, 1)
v_tr_t = torch.tensor(v_train).view(-1, 1)

# 2. Model definition
class AccelNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

model = AccelNet()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

def rk4_step(net, y_t, v_t, u_t, dt):
    def f(y_, v_, u_):
        return net(torch.cat([y_, v_, u_], dim=-1))

    k1_v = f(y_t, v_t, u_t)
    k1_y = v_t
    k2_v = f(y_t + 0.5*dt*k1_y, v_t + 0.5*dt*k1_v, u_t)
    k2_y = v_t + 0.5*dt*k1_v
    k3_v = f(y_t + 0.5*dt*k2_y, v_t + 0.5*dt*k2_v, u_t)
    k3_y = v_t + 0.5*dt*k2_v
    k4_v = f(y_t + dt*k3_y, v_t + dt*k3_v, u_t)
    k4_y = v_t + dt*k3_v

    y_next = y_t + (dt/6.0)*(k1_y + 2*k2_y + 2*k3_y + k4_y)
    v_next = v_t + (dt/6.0)*(k1_v + 2*k2_v + 2*k3_v + k4_v)
    return y_next, v_next

# 3. Fast Training Loop (800 steps)
N_tr = 3072
L = 32
batch_size = 32

for step in range(800):
    starts = np.random.randint(0, N_tr - L, size=batch_size)
    y_curr = y_tr_t[starts]
    v_curr = v_tr_t[starts]

    loss = 0.0
    for i in range(L):
        u_curr = u_tr_t[starts + i]
        y_next, v_next = rk4_step(model, y_curr, v_curr, u_curr, dt)
        y_target = y_tr_t[starts + i + 1]
        loss += torch.mean((y_next - y_target)**2)
        y_curr, v_curr = y_next, v_next

    loss = loss / L
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

# 4. Free-Run Test Evaluation
u_all_t = torch.tensor(u).view(-1, 1)
y_all_t = torch.tensor(y).view(-1, 1)

with torch.no_grad():
    y_p = y_all_t[0:1]
    v_p = torch.tensor(v[0:1]).view(1, 1)
    preds = []
    for t in range(4096):
        preds.append(y_p.item())
        if t < 4095:
            u_p = u_all_t[t:t+1]
            y_p, v_p = rk4_step(model, y_p, v_p, u_p, dt)

y_pred_np = np.array(preds)
test_rmse = np.sqrt(np.mean((y_pred_np[3072:] - y_test)**2))
print(f"REAL TEST RMSE: {test_rmse:.6f}")
'''

    exec_result = execute_python_code(fixed_code)

    return {
        "dialogue": [
            {"role": "Executor (Code Generated)", "content": fixed_code},
            {"role": "Executor (Execution Result)", "content": exec_result}
        ]
    }


# 5. Routing Տրամաբանություն
def router(state: AgentState):
    role = state.get("current_role", "finish")
    if role == "scientist":
        return "scientist"
    elif role == "general":
        return "general"
    elif role == "executor":
        return "executor"
    else:
        return END


# 6. LangGraph Գրաֆի Կառուցում
builder = StateGraph(AgentState)

builder.add_node("orchestrator", orchestrator_node)
builder.add_node("scientist", scientist_node)
builder.add_node("general", general_node)
builder.add_node("executor", executor_node)

builder.set_entry_point("orchestrator")

builder.add_conditional_edges(
    "orchestrator",
    router,
    {"scientist": "scientist", "general": "general", "executor": "executor", END: END}
)

builder.add_edge("scientist", "orchestrator")
builder.add_edge("general", "orchestrator")
builder.add_edge("executor", "orchestrator")

app = builder.compile()
print("--- Agent System Setup Complete ---")

In [ ]:
initial_state = {
    "dialogue": [
        {
            "role": "User",
            "content": "Execute Task 5 on SNLS80mV.mat with vectorised RK4 integration and gradient clipping. Print REAL TEST RMSE."
        }
    ],
    "current_role": "",
    "instruction": "",
    "reason": "",
    "step_count": 0
}

print("=== Starting LangGraph Agentic Pipeline ===")
for output in app.stream(initial_state):
    for node_name, state_update in output.items():
        print(f"\n--- Node: {node_name} ---")
        if "dialogue" in state_update:
            last_msg = state_update["dialogue"][-1]
            print(f"[{last_msg['role']}]: {last_msg['content'][:300]}...")